<a href="https://colab.research.google.com/github/pozdnyavladimer-jpg/geometric-state-navigator/blob/main/research/gsl_text_encoder_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =========================
# GSL TEXT vs CODE ENCODER DEMO
# =========================

import re

def encode_text_to_state(text):
    text_lower = text.lower()

    # базові значення
    v = {
        "red_mass": 0.15,
        "orange_flow": 0.15,
        "yellow_struct": 0.15,
        "green_balance": 0.15,
        "blue_law": 0.15,
        "violet_future": 0.15,
    }

    # --- RED (pressure) ---
    if any(word in text_lower for word in ["error", "fail", "problem", "unstable", "loop"]):
        v["red_mass"] += 0.2

    # --- ORANGE (flow) ---
    if any(word in text_lower for word in ["adapt", "retry", "move", "explore"]):
        v["orange_flow"] += 0.2

    # --- YELLOW (structure) ---
    if any(word in text_lower for word in ["plan", "structure", "step", "define"]):
        v["yellow_struct"] += 0.2

    # --- GREEN (balance) ---
    if "but" in text_lower or "while" in text_lower:
        v["green_balance"] += 0.2

    # --- BLUE (rules / code-like) ---
    if any(sym in text for sym in ["if", "for", "while", "def", "return"]):
        v["blue_law"] += 0.2

    # --- VIOLET (future / potential) ---
    if any(word in text_lower for word in ["future", "potential", "next", "change"]):
        v["violet_future"] += 0.2

    # нормалізація
    total = sum(v.values())
    for k in v:
        v[k] /= total

    return v


def print_state(name, text):
    state = encode_text_to_state(text)

    print(f"\n=== {name} ===")
    print(text)
    print("\nState:")
    for k, val in state.items():
        print(f"{k:15}: {val:.3f}")


# =========================
# DEMO CASES
# =========================

cases = {
    "TEXT (balanced)": "We should reduce instability but keep flexibility and adapt the plan.",

    "TEXT (unstable)": "The system is failing, unstable, and stuck in a loop with errors.",

    "CODE (good structure)": """
def process(data):
    if not data:
        return None
    for item in data:
        handle(item)
    return True
""",

    "CODE (bad loop)": """
while True:
    pass
"""
}

for name, text in cases.items():
    print_state(name, text)


=== TEXT (balanced) ===
We should reduce instability but keep flexibility and adapt the plan.

State:
red_mass       : 0.100
orange_flow    : 0.233
yellow_struct  : 0.233
green_balance  : 0.233
blue_law       : 0.100
violet_future  : 0.100

=== TEXT (unstable) ===
The system is failing, unstable, and stuck in a loop with errors.

State:
red_mass       : 0.318
orange_flow    : 0.136
yellow_struct  : 0.136
green_balance  : 0.136
blue_law       : 0.136
violet_future  : 0.136

=== CODE (good structure) ===

def process(data):
    if not data:
        return None
    for item in data:
        handle(item)
    return True


State:
red_mass       : 0.136
orange_flow    : 0.136
yellow_struct  : 0.136
green_balance  : 0.136
blue_law       : 0.318
violet_future  : 0.136

=== CODE (bad loop) ===

while True:
    pass


State:
red_mass       : 0.115
orange_flow    : 0.115
yellow_struct  : 0.115
green_balance  : 0.269
blue_law       : 0.269
violet_future  : 0.115


In [ ]:

# =========================
# GSL TEXT vs CODE ENCODER DEMO — v2 (code-aware)
# =========================

import re

def normalize_state(v):
    total = sum(max(v[k], 0.0) for k in v)
    if total <= 1e-9:
        return {k: 1.0 / len(v) for k in v}
    return {k: max(v[k], 0.0) / total for k in v}

def looks_like_code(text: str) -> bool:
    code_signals = [
        r"\bdef\b", r"\bclass\b", r"\bif\b", r"\bfor\b", r"\bwhile\b",
        r"\breturn\b", r"\btry\b", r"\bexcept\b", r"\bimport\b",
        r"[{}():=]", r"\bpass\b"
    ]
    hits = sum(bool(re.search(p, text)) for p in code_signals)
    return hits >= 2

def encode_text_to_state_v2(text):
    text_lower = text.lower()

    v = {
        "red_mass": 0.15,
        "orange_flow": 0.15,
        "yellow_struct": 0.15,
        "green_balance": 0.15,
        "blue_law": 0.15,
        "violet_future": 0.15,
    }

    # -------------------------------------------------
    # NATURAL LANGUAGE SIGNALS
    # -------------------------------------------------
    if any(word in text_lower for word in ["error", "fail", "failing", "problem", "unstable", "conflict"]):
        v["red_mass"] += 0.18

    if any(word in text_lower for word in ["loop", "stuck", "deadlock", "blocked"]):
        v["red_mass"] += 0.12
        v["orange_flow"] -= 0.06

    if any(word in text_lower for word in ["adapt", "retry", "move", "explore", "flexibility", "alternative"]):
        v["orange_flow"] += 0.18

    if any(word in text_lower for word in ["plan", "structure", "step", "define", "organize"]):
        v["yellow_struct"] += 0.18

    if any(word in text_lower for word in ["but", "while", "however", "balance", "integrate"]):
        v["green_balance"] += 0.18

    if any(word in text_lower for word in ["rule", "constraint", "validate", "formal", "strict"]):
        v["blue_law"] += 0.18

    if any(word in text_lower for word in ["future", "potential", "next", "change", "transition"]):
        v["violet_future"] += 0.18

    # -------------------------------------------------
    # CODE-SPECIFIC SIGNALS
    # -------------------------------------------------
    if looks_like_code(text):
        # formal syntax / structure
        if any(tok in text for tok in ["def ", "class ", "return", "if ", "for ", "while ", "try:", "except", "import "]):
            v["blue_law"] += 0.14
            v["yellow_struct"] += 0.08

        # good control flow hints
        if "return" in text_lower:
            v["yellow_struct"] += 0.06
            v["green_balance"] += 0.04

        # explicit iteration may increase flow a bit
        if re.search(r"\bfor\b", text_lower):
            v["orange_flow"] += 0.04

        # while True = strong loop pressure
        if re.search(r"while\s+true\s*:", text_lower):
            v["red_mass"] += 0.25
            v["orange_flow"] -= 0.12
            v["green_balance"] -= 0.10

        # pass = inert placeholder, often no real movement
        if re.search(r"\bpass\b", text_lower):
            v["orange_flow"] -= 0.10
            v["yellow_struct"] -= 0.02
            v["red_mass"] += 0.05

        # bare except = risky hidden instability
        if re.search(r"except\s*:", text_lower):
            v["red_mass"] += 0.12
            v["blue_law"] -= 0.05
            v["green_balance"] -= 0.05

        # retry patterns can be good or risky; here treat repeated retries as pressure + flow
        retry_hits = len(re.findall(r"\bretry\b", text_lower))
        if retry_hits >= 1:
            v["orange_flow"] += 0.06
            v["red_mass"] += 0.05 * retry_hits

        # recursion heuristic: function calling itself
        fn_names = re.findall(r"def\s+([a-zA-Z_]\w*)\s*\(", text)
        for fn in fn_names:
            if re.search(rf"\b{fn}\s*\(", text.split(":", 1)[-1]):
                v["violet_future"] += 0.05
                v["red_mass"] += 0.08
                # if no obvious base case marker, penalize more
                if not any(marker in text_lower for marker in ["if ", "return none", "return 0", "return false", "return true"]):
                    v["red_mass"] += 0.08
                    v["green_balance"] -= 0.04

        # very deep branching heuristic
        if text.count("if ") + text.count("elif ") >= 4:
            v["yellow_struct"] += 0.05
            v["red_mass"] += 0.08
            v["green_balance"] -= 0.04

        # handler-like or structured processing is healthy
        if any(word in text_lower for word in ["handle(", "process(", "validate(", "check("]):
            v["yellow_struct"] += 0.05
            v["green_balance"] += 0.04

    return normalize_state(v)

def interpret_state(state):
    red = state["red_mass"]
    orange = state["orange_flow"]
    yellow = state["yellow_struct"]
    green = state["green_balance"]
    blue = state["blue_law"]
    violet = state["violet_future"]

    notes = []

    if red > 0.26:
        notes.append("high pressure / instability")
    elif red < 0.11:
        notes.append("low residual pressure")

    if orange > 0.21:
        notes.append("high adaptability / movement")
    elif orange < 0.11:
        notes.append("low flow / possible rigidity")

    if yellow > 0.21:
        notes.append("strong structure")
    elif yellow < 0.11:
        notes.append("weak structure")

    if green > 0.21:
        notes.append("good internal balance")
    elif green < 0.11:
        notes.append("low balance")

    if blue > 0.23:
        notes.append("strong formal control")
    elif blue < 0.10:
        notes.append("weak constraint discipline")

    if violet > 0.20:
        notes.append("high transition potential")

    return ", ".join(notes) if notes else "mixed state"

def print_state_v2(name, text):
    state = encode_text_to_state_v2(text)

    print(f"\n=== {name} ===")
    print(text)
    print("\nState:")
    for k, val in state.items():
        print(f"{k:15}: {val:.3f}")
    print("Interpretation:", interpret_state(state))

# =========================
# DEMO CASES
# =========================

cases_v2 = {
    "TEXT (balanced)": "We should reduce instability but keep flexibility and adapt the plan.",

    "TEXT (unstable)": "The system is failing, unstable, and stuck in a loop with errors.",

    "CODE (good structure)": """
def process(data):
    if not data:
        return None
    for item in data:
        handle(item)
    return True
""",

    "CODE (bad loop)": """
while True:
    pass
""",

    "CODE (bare except)": """
try:
    risky_action()
except:
    pass
""",

    "CODE (retry pressure)": """
for attempt in range(5):
    if retry:
        retry()
"""
}

for name, text in cases_v2.items():
    print_state_v2(name, text)


=== TEXT (balanced) ===
We should reduce instability but keep flexibility and adapt the plan.

State:
red_mass       : 0.104
orange_flow    : 0.229
yellow_struct  : 0.229
green_balance  : 0.229
blue_law       : 0.104
violet_future  : 0.104
Interpretation: low residual pressure, high adaptability / movement, strong structure, good internal balance

=== TEXT (unstable) ===
The system is failing, unstable, and stuck in a loop with errors.

State:
red_mass       : 0.395
orange_flow    : 0.079
yellow_struct  : 0.132
green_balance  : 0.132
blue_law       : 0.132
violet_future  : 0.132
Interpretation: high pressure / instability, low flow / possible rigidity

=== CODE (good structure) ===

def process(data):
    if not data:
        return None
    for item in data:
        handle(item)
    return True


State:
red_mass       : 0.111
orange_flow    : 0.141
yellow_struct  : 0.252
green_balance  : 0.170
blue_law       : 0.215
violet_future  : 0.111
Interpretation: strong structure

=== CODE (b